# **Tasas de mortalidad**

## **Librerías y modulos necesarios**

In [143]:
import warnings
import itertools
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from unidecode import unidecode
warnings.filterwarnings("ignore")

## **Tasas crudas de mortalidad**

Se calcularon las tasas crudas de mortalidad por cada 100.000 personas, para cada departamento y por año, utilizando la siguiente fórmula:

$$
T_{kj} = \left( \frac{n_{kj}}{p_j} \right) \times k
$$

donde:

- $ T_{kj} $ es la tasa cruda de mortalidad para el año j y el departamento correspondiente.  
- $ n_{kj}$ es el número de muertes registradas.  
- $ p_j $ es la población total en ese año.  
- $ k $ es el factor de estandarización, en este caso $ k = 100.000$.


```{note}
Es importante tener en cuenta que el conjunto de datos utilizado solo registra muertes en mujeres mayores de 40 años, en el periodo comprendido entre los años 2009 y 2023. Por esta razón, las poblaciones utilizadas para el cálculo de las tasas corresponden exclusivamente a mujeres de 40 años o más, dentro de ese intervalo temporal.
```

In [144]:
url_muertes = 'https://github.com/sePerezAlbor/Data/blob/main/data_to_tasas.csv?raw=true'
url_poblacion = 'https://github.com/sePerezAlbor/Data/blob/main/POB-M40-DEP-2009-2023.xlsx?raw=true'

muertes = pd.read_csv(url_muertes).drop(columns=['Unnamed: 0'])
poblacion = pd.read_excel(url_poblacion)

In [145]:
muertes['Nombre_Departamento_Def'].unique()

array(['ANTIOQUIA', 'ATLÁNTICO', 'BOGOTA DC', 'BOLÍVAR', 'BOYACÁ',
       'CALDAS', 'CAQUETÁ', 'CAUCA', 'CESAR', 'CÓRDOBA', 'CUNDINAMARCA',
       'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARIÑO',
       'NORTE DE SANTANDER', 'QUINDIO', 'RISARALDA', 'SANTANDER', 'SUCRE',
       'TOLIMA', 'VALLE DEL CAUCA', 'CASANARE',
       'ARCHIPIÉLAGO DE SAN ANDRÉS Y PROVIDENCIA Y SANTA CATALINA',
       'CHOCÓ', 'ARAUCA', 'GUAVIARE', 'PUTUMAYO', 'VICHADA', 'AMAZONAS',
       'VAUPÉS', 'GUAINÍA'], dtype=object)

In [146]:
poblacion['DPNOM'].unique()

array(['Antioquia', 'Atlántico', 'Bogotá, D.C.', 'Bolívar', 'Boyacá',
       'Caldas', 'Caquetá', 'Cauca', 'Cesar', 'Córdoba', 'Cundinamarca',
       'Chocó', 'Huila', 'La Guajira', 'Magdalena', 'Meta', 'Nariño',
       'Norte de Santander', 'Quindio', 'Risaralda', 'Santander', 'Sucre',
       'Tolima', 'Valle del Cauca', 'Arauca', 'Casanare', 'Putumayo',
       'Archipiélago de San Andrés', 'Amazonas', 'Guainía', 'Guaviare',
       'Vaupés', 'Vichada'], dtype=object)

In [147]:
muertes['Nombre_Departamento_Def'] = muertes['Nombre_Departamento_Def'].str.upper()
muertes['Nombre_Departamento_Def'] = muertes['Nombre_Departamento_Def'].apply(unidecode)

poblacion['DPNOM'] = poblacion['DPNOM'].str.upper()
poblacion['DPNOM'] = poblacion['DPNOM'].apply(unidecode)

In [148]:
equivalencias = {
    'BOGOTA, D.C.': 'BOGOTA DC',
    'ARCHIPIELAGO DE SAN ANDRES Y PROVIDENCIA Y SANTA CATALINA': 
    'ARCHIPIELAGO DE SAN ANDRES'}

muertes['Nombre_Departamento_Def'] = muertes['Nombre_Departamento_Def'].replace(equivalencias)
muertes['Nombre_Departamento_Res'] = muertes['Nombre_Departamento_Res'].replace(equivalencias)

poblacion['DPNOM'] = poblacion['DPNOM'].replace(equivalencias)

In [149]:
muertes['Nombre_Departamento_Def'].unique()

array(['ANTIOQUIA', 'ATLANTICO', 'BOGOTA DC', 'BOLIVAR', 'BOYACA',
       'CALDAS', 'CAQUETA', 'CAUCA', 'CESAR', 'CORDOBA', 'CUNDINAMARCA',
       'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARINO',
       'NORTE DE SANTANDER', 'QUINDIO', 'RISARALDA', 'SANTANDER', 'SUCRE',
       'TOLIMA', 'VALLE DEL CAUCA', 'CASANARE',
       'ARCHIPIELAGO DE SAN ANDRES', 'CHOCO', 'ARAUCA', 'GUAVIARE',
       'PUTUMAYO', 'VICHADA', 'AMAZONAS', 'VAUPES', 'GUAINIA'],
      dtype=object)

In [150]:
poblacion['DPNOM'].unique()

array(['ANTIOQUIA', 'ATLANTICO', 'BOGOTA DC', 'BOLIVAR', 'BOYACA',
       'CALDAS', 'CAQUETA', 'CAUCA', 'CESAR', 'CORDOBA', 'CUNDINAMARCA',
       'CHOCO', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARINO',
       'NORTE DE SANTANDER', 'QUINDIO', 'RISARALDA', 'SANTANDER', 'SUCRE',
       'TOLIMA', 'VALLE DEL CAUCA', 'ARAUCA', 'CASANARE', 'PUTUMAYO',
       'ARCHIPIELAGO DE SAN ANDRES', 'AMAZONAS', 'GUAINIA', 'GUAVIARE',
       'VAUPES', 'VICHADA'], dtype=object)

In [151]:
muertes_df = muertes.copy()

In [152]:
muertes.rename(columns={'Nombre_Departamento_Def': 'DPNOM', 'año_def': 'ANIO'}, inplace=True)

In [153]:
# cond = (muertes['DPNOM'] == 'BOGOTA DC') | (muertes['Nombre_Departamento_Res'] == 'BOGOTA DC')
# muertes.loc[cond, ['DPNOM', 'Nombre_Departamento_Res']] = 'CUNDINAMARCA'

años = list(range(2009, 2024))
departamentos = muertes['DPNOM'].unique()
combinaciones = pd.DataFrame(list(itertools.product(departamentos, años)), columns=['DPNOM', 'ANIO'])

muertes_agg = muertes.groupby(['DPNOM', 'ANIO']).size().reset_index(name='FALLECIDAS')
muertes_completa = combinaciones.merge(muertes_agg, on=['DPNOM', 'ANIO'], how='left').fillna(0)
muertes_completa['FALLECIDAS'] = muertes_completa['FALLECIDAS'].astype(int)

muertes = muertes_completa.copy()

cols_mujeres = [col for col in poblacion.columns if col.startswith('Mujeres_')]
poblacion['TOTAL_MUJERES'] = poblacion[cols_mujeres].replace(',', '', regex=True).astype(float).sum(axis=1)

departamentos_amazona = ['AMAZONAS', 'GUAINIA', 'GUAVIARE', 'VAUPES', 'VICHADA']

def agrupar_grupo(df, cols_sum, nombre_grupo):
    agrupado = df[df['DPNOM'].isin(departamentos_amazona)].groupby('ANIO', as_index=False).sum(numeric_only=True)
    agrupado['DPNOM'] = nombre_grupo
    cols = ['DPNOM', 'ANIO'] + [col for col in agrupado.columns if col not in ['DPNOM', 'ANIO']]
    return agrupado[cols]

amazona_poblacion = agrupar_grupo(poblacion, cols_mujeres + ['TOTAL_MUJERES'], 'GRUPO AMAZONA')
amazona_muertes = agrupar_grupo(muertes, ['FALLECIDAS'], 'GRUPO AMAZONA')

poblacion = pd.concat([poblacion[~poblacion['DPNOM'].isin(departamentos_amazona)], amazona_poblacion], ignore_index=True)
muertes = pd.concat([muertes[~muertes['DPNOM'].isin(departamentos_amazona)], amazona_muertes], ignore_index=True)

def calcular_tasa_mortalidad(muertes, poblacion, año_inicio=2009, año_fin=2023, k=100000):
    muertes_filtrado = muertes[(muertes['ANIO'] >= año_inicio) & (muertes['ANIO'] <= año_fin)]
    pob_filtrada = poblacion[(poblacion['ANIO'] >= año_inicio) & (poblacion['ANIO'] <= año_fin)]

    muertes_agg = muertes_filtrado.groupby('DPNOM', as_index=False)['FALLECIDAS'].sum()
    pob_agg = pob_filtrada.groupby('DPNOM', as_index=False)['TOTAL_MUJERES'].sum()

    tasa = pd.merge(muertes_agg, pob_agg, on='DPNOM')
    tasa['Tasa_Mortalidad'] = (tasa['FALLECIDAS'] / tasa['TOTAL_MUJERES']) * k 

    return tasa


In [154]:
tasa_mortalidad  = calcular_tasa_mortalidad(muertes, poblacion, año_inicio=2009, año_fin=2023, k = 100000)

In [155]:
tasa_mortalidad

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad
0,ANTIOQUIA,6186,19011651.0,32.537942
1,ARAUCA,137,543874.0,25.189658
2,ARCHIPIELAGO DE SAN ANDRES,30,189144.0,15.860931
3,ATLANTICO,3122,6773319.0,46.092617
4,BOGOTA DC,8263,22383858.0,36.914995
5,BOLIVAR,1590,5135499.0,30.960964
6,BOYACA,753,3751470.0,20.072132
7,CALDAS,1014,3393571.0,29.880029
8,CAQUETA,203,852787.0,23.804303
9,CASANARE,159,862417.0,18.436557


In [156]:
mayor = tasa_mortalidad.loc[tasa_mortalidad['Tasa_Mortalidad'].idxmax()]
menor = tasa_mortalidad.loc[tasa_mortalidad['Tasa_Mortalidad'].idxmin()]
promedio = tasa_mortalidad['Tasa_Mortalidad'].mean()

print("\nMayor tasa:", mayor['DPNOM'], f"{mayor['Tasa_Mortalidad']:.2f}")
print("Menor tasa:", menor['DPNOM'], f"{menor['Tasa_Mortalidad']:.2f}")
print("Promedio:", f"{promedio:.2f}")


Mayor tasa: ATLANTICO 46.09
Menor tasa: CHOCO 4.93
Promedio: 25.84


## **Tasas ajustadas por edad**

Para estimar tasas ajustadas de mortalidad por cáncer y poder realizar comparaciones válidas entre diferentes regiones o periodos de tiempo, se utilizó el método directo de ajuste por edad, tomando como referencia la **población estándar de Segi (1960)**. Esta población fue diseñada para facilitar la comparación internacional de tasas de mortalidad y morbilidad, y se basa en una distribución teórica de población que representa una media ponderada de estructuras poblacionales de distintas regiones del mundo en ese momento.  
El ajuste por edad corrige las posibles distorsiones generadas por diferencias en la estructura etaria de las poblaciones observadas, haciendo más precisa la comparación del riesgo de morir por una causa específica, como el cáncer, entre distintos departamentos o periodos.  
La tasa ajustada se expresa por cada **100.000 habitantes**, lo cual es un estándar internacional que facilita la comunicación y análisis de datos epidemiológicos.

Ahmad, O. B., Boschi-Pinto, C., Lopez, A. D., Murray, C. J., Lozano, R., & Inoue, M. (2001). Age standardization of rates: a new WHO standard. Geneva: World Health Organization. GPE Discussion Paper Series: No. 31.


| Grupo de Edad        | Intervalo                   | Proporción Segi (%) |
|----------------------|-----------------------------|----------------------|
| **Grupo_edad1**      | 40–49 años                  | 6.0 + 6.0 = **12.00** |
| **Grupo_edad2**      | 50–59 años                  | 5.0 + 4.0 = **9.00**  |
| **Grupo_edad3**      | 60–69 años                  | 4.0 + 3.0 = **7.00**  |
| **Grupo_edad4**      | 70 años en adelante (70+)   | 2.0 + 1.0 + 0.5 + 0.5 = **4.00** |



In [157]:
def procesar_tasas_por_periodo(muertes_df, año_inicio, año_fin):
    muertes_df = muertes_df[(muertes_df['año_def'] >= año_inicio) & (muertes_df['año_def'] <= año_fin)]
    # Cargar datos de muertes
    columnas_muertes = ['Nombre_Departamento_Def', 'grupo_edad']
    conteos_muertes = muertes_df.groupby(columnas_muertes, observed=False).size().reset_index(name='count')

    # Agrupar departamentos amazónicos
    departamentos_amazona = ['AMAZONAS', 'GUAINIA', 'GUAVIARE', 'VAUPES', 'VICHADA']
    amazona_muertes_df = conteos_muertes[conteos_muertes['Nombre_Departamento_Def'].isin(departamentos_amazona)]
    
    if not amazona_muertes_df.empty:
        amazona_sum_df = amazona_muertes_df.groupby(['grupo_edad'], observed=False)['count'].sum().reset_index()
        amazona_sum_df['Nombre_Departamento_Def'] = 'GRUPO AMAZONA'
        conteos_muertes = conteos_muertes[~conteos_muertes['Nombre_Departamento_Def'].isin(departamentos_amazona)]
        conteos_muertes = pd.concat([conteos_muertes, amazona_sum_df[['Nombre_Departamento_Def', 'grupo_edad', 'count']]], ignore_index=True)

    grupos_edad_estandar = ['40-54 años', '55-64 años', '65-74 años', '75+ años']
    departamentos_unicos = conteos_muertes['Nombre_Departamento_Def'].unique()

    if len(departamentos_unicos) == 0: # No hay datos de muertes para el período
        cols_resumen = ['DPNOM', 'FALLECIDAS', 'TOTAL_MUJERES', 'Tasa_Mortalidad_Cruda', 'TAE']
        cols_detalle = ['DPNOM', 'grupo_edad', 'count', 'poblacion']
        return pd.DataFrame(columns=cols_resumen), pd.DataFrame(columns=cols_detalle)


    all_combinations = pd.DataFrame(list(itertools.product(departamentos_unicos, grupos_edad_estandar)), columns=['Nombre_Departamento_Def', 'grupo_edad'])
    
    # Si los 'grupo_edad' de 'conteos_muertes' no están en el formato estándar, el siguiente merge podría no ser efectivo.
    # Se recomienda pre-procesar 'grupo_edad' en 'muertes_df' para que coincida con 'grupos_edad_estandar'.
    muertes_completas_df = pd.merge(all_combinations, conteos_muertes, on=['Nombre_Departamento_Def', 'grupo_edad'], how='left')
    muertes_completas_df['count'] = muertes_completas_df['count'].fillna(0).astype(int)
    muertes_completas_df.rename(columns={'Nombre_Departamento_Def': 'DPNOM'}, inplace=True)

    # Procesamiento de Población (requiere 'poblacion' y 'cols_mujeres' globales)
    poblacion_df_filtrada = poblacion[(poblacion['ANIO'] >= año_inicio) & (poblacion['ANIO'] <= año_fin)].copy()
    
    for col in cols_mujeres:
        if col in poblacion_df_filtrada.columns:
            poblacion_df_filtrada[col] = pd.to_numeric(poblacion_df_filtrada[col].astype(str).str.replace(',', '', regex=False), errors='coerce')
    
    poblacion_df_filtrada['40-54 años'] = poblacion_df_filtrada[[f'Mujeres_{edad}' for edad in range(40, 55) if f'Mujeres_{edad}' in poblacion_df_filtrada.columns]].sum(axis=1)
    poblacion_df_filtrada['55-64 años'] = poblacion_df_filtrada[[f'Mujeres_{edad}' for edad in range(55, 65) if f'Mujeres_{edad}' in poblacion_df_filtrada.columns]].sum(axis=1)
    poblacion_df_filtrada['65-74 años'] = poblacion_df_filtrada[[f'Mujeres_{edad}' for edad in range(65, 75) if f'Mujeres_{edad}' in poblacion_df_filtrada.columns]].sum(axis=1)
    
    cols_edad_75_mas = [f'Mujeres_{edad}' for edad in range(75, 85) if f'Mujeres_{edad}' in poblacion_df_filtrada.columns]
    if 'Mujeres_85 y más' in poblacion_df_filtrada.columns:
        cols_edad_75_mas.append('Mujeres_85 y más')
    poblacion_df_filtrada['75+ años'] = poblacion_df_filtrada[cols_edad_75_mas].sum(axis=1)

    columnas_pob_interes = ['DPNOM'] + grupos_edad_estandar # DPNOM debe existir en poblacion_df_filtrada
    poblacion_seleccionada_df = poblacion_df_filtrada[columnas_pob_interes]

    poblacion_melted_df = poblacion_seleccionada_df.melt(id_vars=['DPNOM'], value_vars=grupos_edad_estandar,
                                                         var_name='grupo_edad', value_name='poblacion_sum_anios')
    
    poblacion_agrupada_df = poblacion_melted_df.groupby(['DPNOM', 'grupo_edad'], observed=False)['poblacion_sum_anios'].sum().reset_index()
    poblacion_agrupada_df.rename(columns={'poblacion_sum_anios': 'poblacion'}, inplace=True)

    data_final_df = pd.merge(muertes_completas_df, poblacion_agrupada_df, on=['DPNOM', 'grupo_edad'], how='left')
    data_final_df['poblacion'] = data_final_df['poblacion'].fillna(0) 

    tabla_detalle = data_final_df[['DPNOM', 'grupo_edad', 'count', 'poblacion']].copy()

    proporcion_segi = {'40-54 años': 12,'55-64 años': 9,'65-74 años': 7,'75+ años': 4}
    total_p_segi = sum(proporcion_segi.values())
    proporcion_segi_normalizada = {k: v / total_p_segi for k, v in proporcion_segi.items()}

    data_final_df['proporcion_segi'] = data_final_df['grupo_edad'].map(proporcion_segi_normalizada).fillna(0)

    data_final_df['tasa_cruda_estrato'] = 0.0
    mask_poblacion_valida = data_final_df['poblacion'] > 0
    if mask_poblacion_valida.any(): # Solo calcular si hay alguna población válida
        data_final_df.loc[mask_poblacion_valida, 'tasa_cruda_estrato'] = \
            data_final_df.loc[mask_poblacion_valida, 'count'] / data_final_df.loc[mask_poblacion_valida, 'poblacion']
    
    data_final_df['tasa_ajustada_parcial'] = data_final_df['proporcion_segi'] * data_final_df['tasa_cruda_estrato']
    data_final_df['tasa_ajustada_parcial'] = pd.to_numeric(data_final_df['tasa_ajustada_parcial'], errors='coerce').fillna(0)

    tae_por_departamento_df = data_final_df.groupby('DPNOM', observed=False)['tasa_ajustada_parcial'].sum().reset_index(name='TAE')
    tae_por_departamento_df['TAE'] *= 100000

    mortalidad_agregada_df = data_final_df.groupby('DPNOM', observed=False).agg(
        FALLECIDAS=('count', 'sum'),
        TOTAL_MUJERES=('poblacion', 'sum')
    ).reset_index()
    
    mortalidad_agregada_df['Tasa_Mortalidad_Cruda'] = 0.0
    mask_pob_total_valida = mortalidad_agregada_df['TOTAL_MUJERES'] > 0
    if mask_pob_total_valida.any():
        mortalidad_agregada_df.loc[mask_pob_total_valida, 'Tasa_Mortalidad_Cruda'] = \
            (mortalidad_agregada_df.loc[mask_pob_total_valida, 'FALLECIDAS'] / mortalidad_agregada_df.loc[mask_pob_total_valida, 'TOTAL_MUJERES']) * 100000

    resumen_df = pd.merge(mortalidad_agregada_df, tae_por_departamento_df, on='DPNOM', how='left')
    
    return resumen_df, tabla_detalle

In [158]:
tasas, resumen = procesar_tasas_por_periodo( muertes_df, 2009, 2023)

In [159]:
resumen

,DPNOM,grupo_edad,count,poblacion
0,ANTIOQUIA,40-54 años,1284,9466112
1,ANTIOQUIA,55-64 años,1620,4916940
2,ANTIOQUIA,65-74 años,1571,2863150
3,ANTIOQUIA,75+ años,1711,1765449
4,ARAUCA,40-54 años,46,312290
...,...,...,...,...
111,VALLE DEL CAUCA,75+ años,1461,1482090
112,GRUPO AMAZONA,40-54 años,21,311721
113,GRUPO AMAZONA,55-64 años,7,117144
114,GRUPO AMAZONA,65-74 años,4,57190


In [160]:
tasas

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE
0,ANTIOQUIA,6186,19011651,32.537942,38.470215
1,ARAUCA,137,543874,25.189658,30.488575
2,ARCHIPIELAGO DE SAN ANDRES,30,189144,15.860931,26.424814
3,ATLANTICO,3122,6773319,46.092617,54.323739
4,BOGOTA DC,8263,22383858,36.914995,44.346626
5,BOLIVAR,1590,5135499,30.960964,35.741545
6,BOYACA,753,3751470,20.072132,21.659738
7,CALDAS,1014,3393571,29.880029,33.180240
8,CAQUETA,203,852787,23.804303,28.849114
9,CASANARE,159,862417,18.436557,23.606235


## **SMR: Standard Mortality Rate**

$$
\text{SMR} = \frac{\text{Número observado de muertes}}{\text{Número esperado de muertes}} = \frac{y_i}{e_i}
$$

$$
e_i = R_T  \times N_i
$$

$$
N_i \text{:  tamaño de la población i }
$$

$$
R_T = \frac{casos \, totales}{Pob \, total}
$$

### **SMR Por departamento**

Moraga, P. (s.f.). Indirect standardization to calculate expected cases. Recuperado el 24 de abril de 2025, de https://www.paulamoraga.com/presentation-course/#/indirect-standardiz.-to-calculate-expected-cases

In [161]:
tasas.head()

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE
0,ANTIOQUIA,6186,19011651,32.537942,38.470215
1,ARAUCA,137,543874,25.189658,30.488575
2,ARCHIPIELAGO DE SAN ANDRES,30,189144,15.860931,26.424814
3,ATLANTICO,3122,6773319,46.092617,54.323739
4,BOGOTA DC,8263,22383858,36.914995,44.346626


In [162]:
def calcular_SMR(tasas):
    muertes_totales_nal = tasas['FALLECIDAS'].sum()
    mujeres_totales_nal = tasas['TOTAL_MUJERES'].sum()
    tasa_nacional = muertes_totales_nal / mujeres_totales_nal
    tasas['MUERTES_ESPERADAS'] = tasa_nacional * tasas['TOTAL_MUJERES']
    tasas['SMR'] = tasas['FALLECIDAS'] / tasas['MUERTES_ESPERADAS'] * 100
    return tasa_nacional, tasas
    

In [163]:
tasa_nacional, tasas = calcular_SMR(tasas)

In [164]:
tasa_nacional

0.0003112886715858057

In [165]:
tasas

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE,MUERTES_ESPERADAS,SMR
0,ANTIOQUIA,6186,19011651,32.537942,38.470215,5918.111584,104.526586
1,ARAUCA,137,543874,25.189658,30.488575,169.301815,80.920574
2,ARCHIPIELAGO DE SAN ANDRES,30,189144,15.860931,26.424814,58.878384,50.952485
3,ATLANTICO,3122,6773319,46.092617,54.323739,2108.457474,148.070333
4,BOGOTA DC,8263,22383858,36.914995,44.346626,6967.841422,118.587659
5,BOLIVAR,1590,5135499,30.960964,35.741545,1598.622662,99.460619
6,BOYACA,753,3751470,20.072132,21.659738,1167.790113,64.480765
7,CALDAS,1014,3393571,29.880029,33.180240,1056.380209,95.988167
8,CAQUETA,203,852787,23.804303,28.849114,265.462932,76.470187
9,CASANARE,159,862417,18.436557,23.606235,268.460642,59.226559


In [166]:
tasas.to_excel('../data/processed/tasas_final.xlsx', index=False)

### **Interpretación del SMR**

- **SMR = 100**: Mortalidad observada igual a la esperada.
- **SMR > 100**: Más muertes de las esperadas (mayor riesgo).
- **SMR < 100**: Menos muertes de las esperadas (menor riesgo).


## **Tasas por quinquenios**

### **`2009 - 2013`**

In [167]:
tasas_q1, resumen_q1 = procesar_tasas_por_periodo(muertes_df, 2009, 2013)

In [168]:
tasas_q1

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE
0,ANTIOQUIA,1812,5568764,32.538639,39.408821
1,ARAUCA,41,144028,28.466687,30.798612
2,ARCHIPIELAGO DE SAN ANDRES,11,56899,19.332501,30.145098
3,ATLANTICO,737,1957050,37.658721,44.531080
4,BOGOTA DC,2395,6557641,36.522280,44.694677
5,BOLIVAR,381,1509793,25.235247,28.481294
6,BOYACA,210,1133291,18.530104,19.601898
7,CALDAS,312,1037805,30.063451,34.757235
8,CAQUETA,58,248410,23.348496,26.127491
9,CASANARE,34,225464,15.080013,18.376796


In [169]:
resumen_q1

,DPNOM,grupo_edad,count,poblacion
0,ANTIOQUIA,40-54 años,545,3050743
1,ANTIOQUIA,55-64 años,442,1322709
2,ANTIOQUIA,65-74 años,401,716429
3,ANTIOQUIA,75+ años,424,478883
4,ARAUCA,40-54 años,22,87154
...,...,...,...,...
111,VALLE DEL CAUCA,75+ años,332,404092
112,GRUPO AMAZONA,40-54 años,4,87313
113,GRUPO AMAZONA,55-64 años,0,28968
114,GRUPO AMAZONA,65-74 años,1,13087


In [170]:
tasa_nacional_q1, tasas_q1 = calcular_SMR(tasas_q1)

In [171]:
tasa_nacional_q1

0.00028685431629910747

In [172]:
tasas_q1

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE,MUERTES_ESPERADAS,SMR
0,ANTIOQUIA,1812,5568764,32.538639,39.408821,1597.423990,113.432627
1,ARAUCA,41,144028,28.466687,30.798612,41.315053,99.237437
2,ARCHIPIELAGO DE SAN ANDRES,11,56899,19.332501,30.145098,16.321724,67.394842
3,ATLANTICO,737,1957050,37.658721,44.531080,561.388240,131.281696
4,BOGOTA DC,2395,6557641,36.522280,44.694677,1881.087626,127.319959
5,BOLIVAR,381,1509793,25.235247,28.481294,433.090639,87.972347
6,BOYACA,210,1133291,18.530104,19.601898,325.089415,64.597612
7,CALDAS,312,1037805,30.063451,34.757235,297.698844,104.803901
8,CAQUETA,58,248410,23.348496,26.127491,71.257481,81.394963
9,CASANARE,34,225464,15.080013,18.376796,64.675322,52.570284


In [173]:
tasas_q1.to_excel('../data/processed/tasas_quinquenio1.xlsx', index=False)

### **`2014 - 2018`**

In [174]:
tasas_q2, resumen_q2 = procesar_tasas_por_periodo(muertes_df, 2014, 2018)

In [175]:
tasas_q2

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE
0,ANTIOQUIA,2058,6271904,32.813002,39.483922
1,ARAUCA,42,171568,24.480090,33.043907
2,ARCHIPIELAGO DE SAN ANDRES,7,62611,11.180144,22.015924
3,ATLANTICO,1054,2213411,47.618811,56.860635
4,BOGOTA DC,2781,7383038,37.667421,46.342932
5,BOLIVAR,498,1686827,29.522885,34.095836
6,BOYACA,253,1239725,20.407752,22.167494
7,CALDAS,352,1125252,31.281882,35.266896
8,CAQUETA,74,282065,26.235088,34.491997
9,CASANARE,39,282462,13.807167,19.606517


In [176]:
resumen_q2

,DPNOM,grupo_edad,count,poblacion
0,ANTIOQUIA,40-54 años,377,3133634
1,ANTIOQUIA,55-64 años,580,1637540
2,ANTIOQUIA,65-74 años,524,935257
3,ANTIOQUIA,75+ años,577,565473
4,ARAUCA,40-54 años,8,99818
...,...,...,...,...
111,VALLE DEL CAUCA,75+ años,507,485880
112,GRUPO AMAZONA,40-54 años,8,101861
113,GRUPO AMAZONA,55-64 años,3,37743
114,GRUPO AMAZONA,65-74 años,1,18165


In [177]:
tasa_nacional_q2, tasas_q2 = calcular_SMR(tasas_q2)

In [178]:
tasa_nacional_q2

0.00031508500096846174

In [179]:
tasas_q2

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE,MUERTES_ESPERADAS,SMR
0,ANTIOQUIA,2058,6271904,32.813002,39.483922,1976.182878,104.140159
1,ARAUCA,42,171568,24.480090,33.043907,54.058503,77.693605
2,ARCHIPIELAGO DE SAN ANDRES,7,62611,11.180144,22.015924,19.727787,35.482946
3,ATLANTICO,1054,2213411,47.618811,56.860635,697.412607,151.130047
4,BOGOTA DC,2781,7383038,37.667421,46.342932,2326.284535,119.546855
5,BOLIVAR,498,1686827,29.522885,34.095836,531.493887,93.698161
6,BOYACA,253,1239725,20.407752,22.167494,390.618753,64.769036
7,CALDAS,352,1125252,31.281882,35.266896,354.550028,99.280771
8,CAQUETA,74,282065,26.235088,34.491997,88.874451,83.263524
9,CASANARE,39,282462,13.807167,19.606517,88.999540,43.820451


In [180]:
tasas_q2.to_excel('../data/processed/tasas_quinquenio2.xlsx', index=False)

### **`2019 - 2023`**

In [181]:
tasas_q3, resumen_q3 = procesar_tasas_por_periodo(muertes_df, 2014, 2018)

In [182]:
tasas_q3

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE
0,ANTIOQUIA,2058,6271904,32.813002,39.483922
1,ARAUCA,42,171568,24.480090,33.043907
2,ARCHIPIELAGO DE SAN ANDRES,7,62611,11.180144,22.015924
3,ATLANTICO,1054,2213411,47.618811,56.860635
4,BOGOTA DC,2781,7383038,37.667421,46.342932
5,BOLIVAR,498,1686827,29.522885,34.095836
6,BOYACA,253,1239725,20.407752,22.167494
7,CALDAS,352,1125252,31.281882,35.266896
8,CAQUETA,74,282065,26.235088,34.491997
9,CASANARE,39,282462,13.807167,19.606517


In [183]:
resumen_q3

,DPNOM,grupo_edad,count,poblacion
0,ANTIOQUIA,40-54 años,377,3133634
1,ANTIOQUIA,55-64 años,580,1637540
2,ANTIOQUIA,65-74 años,524,935257
3,ANTIOQUIA,75+ años,577,565473
4,ARAUCA,40-54 años,8,99818
...,...,...,...,...
111,VALLE DEL CAUCA,75+ años,507,485880
112,GRUPO AMAZONA,40-54 años,8,101861
113,GRUPO AMAZONA,55-64 años,3,37743
114,GRUPO AMAZONA,65-74 años,1,18165


In [184]:
tasa_nacional_q3, tasas_q3 = calcular_SMR(tasas_q3)

In [185]:
tasa_nacional_q3

0.00031508500096846174

In [186]:
tasas_q3

,DPNOM,FALLECIDAS,TOTAL_MUJERES,Tasa_Mortalidad_Cruda,TAE,MUERTES_ESPERADAS,SMR
0,ANTIOQUIA,2058,6271904,32.813002,39.483922,1976.182878,104.140159
1,ARAUCA,42,171568,24.480090,33.043907,54.058503,77.693605
2,ARCHIPIELAGO DE SAN ANDRES,7,62611,11.180144,22.015924,19.727787,35.482946
3,ATLANTICO,1054,2213411,47.618811,56.860635,697.412607,151.130047
4,BOGOTA DC,2781,7383038,37.667421,46.342932,2326.284535,119.546855
5,BOLIVAR,498,1686827,29.522885,34.095836,531.493887,93.698161
6,BOYACA,253,1239725,20.407752,22.167494,390.618753,64.769036
7,CALDAS,352,1125252,31.281882,35.266896,354.550028,99.280771
8,CAQUETA,74,282065,26.235088,34.491997,88.874451,83.263524
9,CASANARE,39,282462,13.807167,19.606517,88.999540,43.820451


In [187]:
tasas_q3.to_excel('../data/processed/tasas_quinquenio3.xlsx', index=False)